# 02 - PySpark SDN DDoS Dataset Analytics

This notebook loads the generated CSV with PySpark, cleans the data, applies numerical and categorical preprocessing, and creates five Plotly visualizations. Each chart includes a short purpose and interpretation.

The dataset is a controlled local-lab profile. It is suitable for exploratory analysis and pipeline testing, not claims about live Internet traffic.

In [ ]:
# Install packages once if necessary.
# Spark itself should be installed in the WSL environment.
#
# %pip install -q pyspark plotly pandas pyarrow

import sys, os
print(sys.version)


In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import DoubleType, IntegerType, LongType, StringType, TimestampType
from pyspark.ml import Pipeline
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler, StandardScaler

spark = (
    SparkSession.builder
    .appName("SDN-DDoS-Analytics")
    .master("local[*]")
    .config("spark.sql.shuffle.partitions", "8")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")
print("Spark:", spark.version)


In [ ]:
from pathlib import Path

DATA_PATH = Path.cwd() / "data" / "processed" / "sdn_ddos_controlled_lab.csv"

if not DATA_PATH.exists():
    raise FileNotFoundError(f"Dataset not found: {DATA_PATH}. Run Network and Traffic/scripts/run_experiment.sh first.")

df = spark.read.option("header", True).option("inferSchema", True).csv(str(DATA_PATH))

print("Rows:", df.count())
print("Columns:", len(df.columns))
df.printSchema()
df.show(5, truncate=False)


## 1. Data cleaning

Cleaning steps:

- Remove exact duplicate rows.
- Cast numeric fields to appropriate numeric types.
- Parse timestamps.
- Replace invalid negative measurements with null.
- Remove records missing essential identifiers/labels.
- Check consistency of packet/byte rates.

We keep the cleaning logic explicit so it can be documented in the project report.


In [ ]:
raw_count = df.count()

clean = df.dropDuplicates()

numeric_cols = [
    "window_seconds", "packet_count", "byte_count", "flow_count",
    "packet_rate", "byte_rate", "flow_rate", "mean_packet_size",
    "tcp_syn_count", "tcp_ack_count", "tcp_handshake_completion_ratio",
    "flow_table_size", "packet_in_count", "controller_cpu_proxy_pct",
    "attacker_count", "background_traffic_level"
]

for col in numeric_cols:
    clean = clean.withColumn(col, F.col(col).cast("double"))

clean = clean.dropna(subset=["scenario_id", "timestamp_utc", "protocol", "label"])
clean = clean.filter(
    (F.col("packet_count") >= 0) &
    (F.col("byte_count") >= 0) &
    (F.col("packet_rate") >= 0) &
    F.col("tcp_handshake_completion_ratio").between(0, 1)
)

clean_count = clean.count()

print("Raw rows:", raw_count)
print("Clean rows:", clean_count)
print("Removed:", raw_count - clean_count)


In [ ]:
# Missing-value profile
missing = clean.select([
    F.sum(F.col(c).isNull().cast("int")).alias(c)
    for c in clean.columns
])

missing.show(truncate=False)


In [ ]:
# Exploratory data analysis
# Spark summaries inspect data quality, distributions, class composition,
# and relationships between the main traffic features.

In [ ]:
# Dataset shape, time span, class balance, protocol composition, and feature statistics.
eda_overview = clean.select(
    F.count("*").alias("rows"),
    F.countDistinct("label").alias("classes"),
    F.countDistinct("protocol").alias("protocols"),
    F.min("timestamp_utc").alias("first_timestamp"),
    F.max("timestamp_utc").alias("last_timestamp")
)
eda_overview.show(truncate=False)

print("Schema after cleaning:")
clean.printSchema()

summary_cols = [
    "packet_rate", "byte_rate", "flow_rate", "mean_packet_size",
    "tcp_handshake_completion_ratio", "flow_table_size",
    "packet_in_count", "controller_cpu_proxy_pct"
]
clean.select(summary_cols).summary(
    "count", "mean", "stddev", "min", "25%", "50%", "75%", "max"
).show()

class_summary = (
    clean.groupBy("label", "attack_family", "protocol")
    .agg(
        F.count("*").alias("records"),
        F.round(100 * F.count("*") / clean_count, 2).alias("share_pct"),
        F.round(F.avg("packet_rate"), 2).alias("mean_packet_rate"),
        F.round(F.avg("byte_rate"), 2).alias("mean_byte_rate"),
        F.round(F.avg("controller_cpu_proxy_pct"), 2).alias("mean_controller_pressure")
    )
    .orderBy(F.desc("records"), "label")
)
class_summary.show(20, truncate=False)

protocol_summary = (
    clean.groupBy("protocol", "label")
    .count()
    .orderBy("protocol", "label")
)
protocol_summary.show(30, truncate=False)

missing_profile = clean.select([
    F.sum(F.col(column).isNull().cast("int")).alias(column)
    for column in clean.columns
])
missing_profile.show(truncate=False)

correlation_cols = [
    "packet_rate", "byte_rate", "flow_rate", "mean_packet_size",
    "flow_table_size", "packet_in_count", "controller_cpu_proxy_pct"
]
correlations = {
    f"{left}__{right}": round(clean.stat.corr(left, right), 3)
    for index, left in enumerate(correlation_cols)
    for right in correlation_cols[index + 1:]
}
print("Selected numeric feature correlations:")
for pair, value in correlations.items():
    print(f"{pair}: {value}")

## 2. Preprocessing

We separate:

### Numerical features
Scaled with `StandardScaler` after vector assembly.

### Categorical features
`protocol` and `attack_intensity` are converted using `StringIndexer` + `OneHotEncoder`.

### Identifier columns
IPs and experiment identifiers are retained for analysis but are **not automatically treated as ML features**, because raw addresses can cause severe leakage and memorization.

The same principle applies to `scenario_id`: it is metadata, not a predictive feature.


In [ ]:
numeric_features = [
    "packet_rate",
    "byte_rate",
    "flow_rate",
    "mean_packet_size",
    "tcp_syn_count",
    "tcp_ack_count",
    "tcp_handshake_completion_ratio",
    "flow_table_size",
    "packet_in_count",
    "controller_cpu_proxy_pct",
    "attacker_count",
    "background_traffic_level",
]

categorical_features = ["protocol", "attack_intensity"]

indexers = [
    StringIndexer(inputCol=x, outputCol=f"{x}_idx", handleInvalid="keep")
    for x in categorical_features
]

encoders = [
    OneHotEncoder(inputCol=f"{x}_idx", outputCol=f"{x}_ohe")
    for x in categorical_features
]

assembler = VectorAssembler(
    inputCols=numeric_features + [f"{x}_ohe" for x in categorical_features],
    outputCol="features_raw",
    handleInvalid="keep"
)

scaler = StandardScaler(
    inputCol="features_raw",
    outputCol="features_scaled",
    withStd=True,
    withMean=False
)

preprocess_pipeline = Pipeline(stages=indexers + encoders + [assembler, scaler])

processed = preprocess_pipeline.fit(clean).transform(clean)

processed.select(
    "label", "protocol", "features_scaled"
).show(5, truncate=False)


## Visualization 1 - Class distribution

Purpose: check whether the dataset contains all traffic classes and whether the classes are balanced.

Interpretation: equal counts indicate a balanced development dataset; production traffic would usually be much more benign-heavy.

In [ ]:
import plotly.express as px
import pandas as pd

class_pd = (
    clean.groupBy("label")
    .count()
    .orderBy(F.desc("count"))
    .toPandas()
)

fig1 = px.bar(
    class_pd,
    x="label",
    y="count",
    color="label",
    title="SDN DDoS Dataset - Class Distribution",
    labels={"label":"Traffic class", "count":"Observations"}
)
fig1.update_layout(showlegend=False, xaxis_tickangle=-45)
fig1.show()


## Visualization 2 - Packet-rate behavior

Purpose: compare traffic intensity across labels using packets per second.

Interpretation: high-rate flood profiles should separate from benign traffic, while low-rate attacks demonstrate why packet rate alone is insufficient.

In [ ]:
packet_pd = (
    clean
    .select("label", "attack_family", "packet_rate")
    .sample(False, 0.10, seed=42)
    .limit(12000)
    .toPandas()
)

fig2 = px.box(
    packet_pd,
    x="label",
    y="packet_rate",
    color="attack_family",
    points=False,
    title="Packet-Rate Distribution by Traffic Class",
    labels={
        "label": "Traffic class",
        "attack_family": "Attack family",
        "packet_rate": "Packets/second"
    }
)
fig2.update_layout(xaxis_tickangle=-45)
fig2.show()

## Visualization 3 - Bubble chart: traffic intensity and flow-table pressure

Purpose: compare packet rate and byte rate while using bubble size for flow-table size and color for traffic class.

Interpretation: the chart exposes traffic classes that combine high volume with high flow-table pressure, a useful SDN-specific view of attack behavior.

In [ ]:
bubble_pd = (
    clean
    .select(
        "label", "attack_family", "packet_rate", "byte_rate",
        "flow_table_size", "controller_cpu_proxy_pct"
    )
    .sample(False, 0.25, seed=42)
    .limit(12000)
    .toPandas()
)

fig3 = px.scatter(
    bubble_pd,
    x="packet_rate",
    y="byte_rate",
    size="flow_table_size",
    color="label",
    hover_data=["attack_family", "controller_cpu_proxy_pct"],
    size_max=32,
    opacity=0.65,
    title="Bubble Chart: Traffic Intensity and Flow-Table Size",
    labels={
        "packet_rate": "Packets/second",
        "byte_rate": "Bytes/second",
        "flow_table_size": "Flow-table entries",
        "label": "Traffic class"
    }
)
fig3.show()

## Visualization 4 - TCP handshake completion

Purpose: examine whether TCP connection attempts complete successfully across TCP traffic classes.

Interpretation: SYN-flood and slow-rate profiles should show lower completion than normal TCP; non-TCP records are excluded.

In [ ]:
tcp_pd = (
    clean
    .filter(F.col("protocol") == "TCP")
    .groupBy("label")
    .agg(F.avg("tcp_handshake_completion_ratio").alias("completion_ratio"))
    .orderBy("completion_ratio")
    .toPandas()
)

fig4 = px.bar(
    tcp_pd,
    x="label",
    y="completion_ratio",
    color="label",
    title="TCP Connection Completion Ratio",
    labels={"label":"Traffic class", "completion_ratio":"Completion ratio"}
)
fig4.update_layout(showlegend=False, xaxis_tickangle=-45)
fig4.show()


## Visualization 5 - SDN control-plane pressure

Purpose: compare controller CPU proxy and Packet-In activity across traffic classes.

Interpretation: Packet-In flood should create the strongest control-plane signal, showing why SDN telemetry complements ordinary packet statistics.

In [ ]:
control_pd = (
    clean.groupBy("label")
    .agg(
        F.avg("controller_cpu_proxy_pct").alias("mean_controller_load"),
        F.avg("packet_in_count").alias("mean_packet_in")
    )
    .orderBy(F.desc("mean_controller_load"))
    .toPandas()
)

fig5 = px.bar(
    control_pd,
    x="label",
    y="mean_controller_load",
    color="label",
    title="Mean SDN Controller-Load Indicator by Traffic Class",
    labels={"label":"Traffic class", "mean_controller_load":"Controller-load indicator"}
)
fig5.update_layout(showlegend=False, xaxis_tickangle=-45)
fig5.show()


# Analytical summary

The EDA combines dataset quality, class balance, protocol composition, numerical distributions, and feature relationships. The five visualizations then focus on:

1. Class balance.
2. Packet-rate distribution by traffic class.
3. Packet rate, byte rate, and flow-table pressure in a bubble chart.
4. TCP handshake completion behavior.
5. SDN controller pressure and Packet-In activity.

Together, these views connect the dataset's network-volume, protocol, flow, and SDN control-plane signals. The scaled Spark feature vector is ready for downstream model development, while the raw identifiers and labels remain excluded from preprocessing features to reduce leakage.